<a href="https://colab.research.google.com/github/ElizabethWaithera/Data_sciences_projects/blob/main/srhr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kenya Demographic and Health Survey (KDHS) - Sexual and Reproductive Health Rights Analysis

## Executive Summary

This report presents a comprehensive analysis of Sexual and Reproductive Health Rights (SRHR) indicators from the Kenya Demographic and Health Survey (KDHS). Our analysis focuses on three key domains: contraceptive use patterns, maternal health care utilization, and age at first marriage/cohabitation.

Key findings reveal significant disparities in SRHR outcomes based on education level, wealth status, geographic region, and urban-rural residence. Modern contraceptive prevalence is approximately 34.3% nationwide, with substantial regional variation. Education emerges as the strongest predictor of positive SRHR outcomes across all domains, with higher education associated with increased contraceptive use, better maternal health care utilization, and delayed marriage.

The analysis demonstrates the need for targeted interventions that address regional disparities and socioeconomic barriers to SRHR services, with particular attention to vulnerable populations including rural residents, those with lower education levels, and adolescents.

In [2]:
!pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 29.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf


In [4]:
# Set style for visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12

### Data Processing

The analysis used the KDHS dataset in Stata format (Contraceptives_2025.dta). Data processing included:
- Converting categorical variables to appropriate formats
- Handling missing values (notable in maternal health indicators)
- Creating derived variables for multivariate analysis
- Data validation and quality checks

In [6]:
# Load the data
print("Loading KDHS dataset...")
df, meta = pyreadstat.read_dta("/content/Contraceptives_2025 (2).dta")

Loading KDHS dataset...


In [7]:
# Get value labels for categorical variables if available
value_labels = {}
if hasattr(meta, 'value_labels'):
    value_labels = meta.value_labels

In [8]:
# ====== DATA EXPLORATION ======
print("\n===== INITIAL DATA EXPLORATION =====")
print(f"Dataset shape: {df.shape} (rows, columns)")


===== INITIAL DATA EXPLORATION =====
Dataset shape: (77613, 27) (rows, columns)


In [9]:
key_srhr_vars = [
    'v313',  # current use by method type
    'v106',  # highest educational level
    'v024',  # region
    'v025',  # type of place of residence
    'v013',  # age in 5-year groups
    'v190',  # wealth index
    'm14_1',  # number of antenatal visits
    'm15_1',  # place of delivery
    'v511',  # age at first cohabitation
    'v701',  # husband/partner's education level
]

In [10]:
# Check data types and missing values for key variables
print("\nKey SRHR variables:")
for var in key_srhr_vars:
    if var in df.columns:
        missing = df[var].isnull().sum()
        missing_pct = (missing / len(df)) * 100
        print(f"{var}: {meta.column_labels[meta.column_names.index(var)]}")
        print(f"  - Data type: {df[var].dtype}")
        print(f"  - Missing values: {missing} ({missing_pct:.2f}%)")

        # For categorical variables, show value counts
        if df[var].dtype == 'object' or (df[var].dtype == 'int64' and df[var].nunique() < 10):
            value_counts = df[var].value_counts(dropna=False).sort_index()
            print(f"  - Value counts: {value_counts.to_dict()}")
        print()


Key SRHR variables:
v313: current use by method type
  - Data type: int64
  - Missing values: 0 (0.00%)
  - Value counts: {0: 48073, 1: 179, 2: 2732, 3: 26629}

v106: highest educational level
  - Data type: int64
  - Missing values: 0 (0.00%)
  - Value counts: {0: 10359, 1: 35314, 2: 23515, 3: 8425}

v024: region
  - Data type: int64
  - Missing values: 0 (0.00%)

v025: type of place of residence
  - Data type: int64
  - Missing values: 0 (0.00%)
  - Value counts: {1: 28445, 2: 49168}

v013: age in 5-year groups
  - Data type: int64
  - Missing values: 0 (0.00%)
  - Value counts: {1: 15458, 2: 13950, 3: 13810, 4: 11056, 5: 9867, 6: 7483, 7: 5989}

v190: wealth index combined
  - Data type: int64
  - Missing values: 0 (0.00%)
  - Value counts: {1: 17088, 2: 13965, 3: 14661, 4: 15782, 5: 16117}

m14_1: number of antenatal visits during pregnancy
  - Data type: object
  - Missing values: 45104 (58.11%)
  - Value counts: {0: 2122, 1: 1203, 2: 3129, 3: 7831, 4: 7495, 5: 5005, 6: 2922, 7: 

### Analytical Approach

The analytical approach combined:
- Descriptive statistics to establish prevalence and patterns
- Bivariate analysis to examine relationships between socioeconomic factors and SRHR outcomes
- Cross-tabulations to analyze patterns across demographic segments
- Logistic regression modeling to identify predictors of modern contraceptive use

In [11]:
# ====== SRHR ANALYSIS ======
print("\n===== SEXUAL AND REPRODUCTIVE HEALTH ANALYSIS =====")

# 1. Contraceptive Use Analysis (v313)
print("\n1. CONTRACEPTIVE USE ANALYSIS:")
if 'v313' in df.columns:
    # Create more readable contraceptive use categories
    contraceptive_map = {
        0: 'Not using',
        1: 'Traditional method',
        2: 'Folk method',
        3: 'Modern method'
    }

    # Map if it's a categorical variable with few unique values
    if df['v313'].nunique() <= 10:
        df['contraceptive_use'] = df['v313'].map(contraceptive_map)

    # Contraceptive use by region
    if 'v024' in df.columns:
        region_contraceptive = pd.crosstab(
            df['v024'],
            df['v313'],
            normalize='index'
        ) * 100

        print("\nContraceptive use by region (%):")
        print(region_contraceptive)

    # Contraceptive use by education level
    if 'v106' in df.columns:
        edu_contraceptive = pd.crosstab(
            df['v106'],
            df['v313'],
            normalize='index'
        ) * 100

        print("\nContraceptive use by education level (%):")
        print(edu_contraceptive)

    # Contraceptive use by wealth index
    if 'v190' in df.columns:
        wealth_contraceptive = pd.crosstab(
            df['v190'],
            df['v313'],
            normalize='index'
        ) * 100

        print("\nContraceptive use by wealth index (%):")
        print(wealth_contraceptive)



===== SEXUAL AND REPRODUCTIVE HEALTH ANALYSIS =====

1. CONTRACEPTIVE USE ANALYSIS:

Contraceptive use by region (%):
v313          0         1         2          3
v024                                          
1     61.843169  0.431151  3.476152  34.249528
2     54.464286  0.235849  3.436658  41.863208
3     70.237548  0.137931  3.509579  26.114943
4     60.348335  0.216120  4.118993  35.316552
5     63.552725  0.283086  1.995754  34.168436
6     64.893348  0.320378  4.426271  30.360003
7     66.958628  0.354081  1.994037  30.693254
8     97.540741  0.000000  0.059259   2.400000
9     98.589563  0.000000  0.141044   1.269394
10    94.183865  0.000000  0.187617   5.628518
11    79.632721  0.000000  0.667780  19.699499
12    41.889483  0.534759  5.704100  51.871658
13    43.712575  0.000000  4.990020  51.297405
14    36.125654  0.174520  6.282723  57.417103
15    52.466368  0.298954  2.092676  45.142003
16    45.494830  1.033973  7.090103  46.381093
17    51.005747  0.431034  5.890805

## Contraceptive Use Patterns

### Overall Prevalence:

About 34.3% of women use modern contraceptive methods, while 61.9% are not using any method. This indicates substantial room for improving contraceptive uptake.

### Regional Disparities:
 There's significant variation in modern contraceptive use across regions:
Highest uptake (50-57%): Regions 14, 19, 20, 22, 27, 28
Lowest uptake (<10%): Regions 8, 9 (less than 3% modern method use)

These disparities suggest the need for targeted interventions in low-uptake regions.

### Education Impact:
Education strongly influences contraceptive use:
Women with higher education: 44.2% use modern methods
Women with no education: Only 9.9% use modern methods
This fourfold difference highlights education's crucial role in contraceptive adoption.

### Wealth Gradient:

Clear socioeconomic disparities exist:
Richest quintile: 38.5% modern method use
Poorest quintile: 19.5% modern method use
Middle-wealth women have similar rates to the richest, suggesting diminishing returns above middle wealth.
Urban-Rural Divide: Urban women have slightly higher modern contraceptive use (39.8%) compared to rural women (36.6%).

In [12]:
# 2. Maternal Health Care Analysis
print("\n2. MATERNAL HEALTH CARE ANALYSIS:")

# Antenatal care visits
if 'm14_1' in df.columns and df['m14_1'].notna().any():
    print("\nAntenatal care visits statistics:")
    anc_stats = df['m14_1'].describe()
    print(anc_stats)

    # ANC visits by education level
    if 'v106' in df.columns:
        try:
            anc_by_edu = df.groupby('v106')['m14_1'].mean()
            print("\nAverage ANC visits by education level:")
            print(anc_by_edu)
        except:
            print("Could not calculate ANC visits by education (might be non-numeric)")

# Place of delivery
if 'm15_1' in df.columns and df['m15_1'].notna().any():
    print("\nPlace of delivery distribution:")
    delivery_counts = df['m15_1'].value_counts(normalize=True) * 100
    print(delivery_counts)

    # Place of delivery by residence (urban/rural)
    if 'v025' in df.columns:
        delivery_by_residence = pd.crosstab(
            df['v025'],
            df['m15_1'],
            normalize='index'
        ) * 100

        print("\nPlace of delivery by residence (urban/rural) (%):")
        print(delivery_by_residence)

# 3. Age at first marriage/cohabitation
if 'v511' in df.columns and df['v511'].notna().any():
    print("\n3. AGE AT FIRST MARRIAGE/COHABITATION:")
    marriage_stats = df['v511'].describe()
    print(marriage_stats)

    # Age at first marriage by education
    if 'v106' in df.columns:
        try:
            marriage_by_edu = df.groupby('v106')['v511'].mean()
            print("\nAverage age at first marriage by education level:")
            print(marriage_by_edu)
        except:
            print("Could not calculate age at marriage by education (might be non-numeric)")


2. MATERNAL HEALTH CARE ANALYSIS:

Antenatal care visits statistics:
count     32509
unique       29
top           3
freq       7831
Name: m14_1, dtype: int64

Average ANC visits by education level:
v106
0    3.574553
1    4.441237
2    4.865434
3    5.777179
Name: m14_1, dtype: object

Place of delivery distribution:
m15_1
11    33.301524
21    32.011698
22     9.875327
31     8.545483
23     5.818070
12     3.530860
32     2.862860
43     1.437587
33     1.219024
96     0.735724
24     0.135447
13     0.126212
35     0.095429
41     0.083115
44     0.070802
26     0.064645
36     0.061567
99     0.024627
Name: proportion, dtype: float64

Place of delivery by residence (urban/rural) (%):
m15_1         11        12        13         21         22        23  \
v025                                                                   
1      14.866260  2.449554  0.018771  48.268419   7.883623  2.674801   
2      42.299588  4.058635  0.178653  24.076958  10.847458  7.352268   

m15_1       

## Maternal Health Care
### Antenatal Care:
Average ANC visits increase with education level (5.8 visits for higher education vs 3.6 visits for no education)
Most women receive 3-4 ANC visits, but many don't reach the WHO-recommended eight visits.

### Delivery Services:
Significant urban-rural disparity in facility deliveries (48.3% of urban deliveries occur in public hospitals [code 21] compared to 24.1% in rural areas)
Home deliveries (code 11) are much more common in rural areas (42.3%) than urban areas (14.9%)
This suggests barriers to accessing facility-based delivery in rural areas.


## Age at First Marriage/Cohabitation

### Marriage Age Distribution:
Modal age at first marriage is 18 years
A substantial proportion marry before age 18, indicating child marriage remains an issue
The red line in the age distribution visualization marks the legal age threshold

### Education and Marriage Age:
Strong positive correlation between education and marriage age:
No education: average age 17.5 years
Higher education: average age 23.1 years
This 5.6-year difference demonstrates education's protective effect against early marriage

In [13]:
# ====== PREDICTIVE MODELING ======
print("\n===== PREDICTIVE MODELING FOR SRHR OUTCOMES =====")

# Model 1: Predicting contraceptive use (logistic regression)
print("\nMODEL 1: FACTORS AFFECTING MODERN CONTRACEPTIVE USE")

# Create binary outcome: using modern method (1) vs not using modern method (0)
if 'v313' in df.columns:
    try:
        # Assuming modern method is coded as value 3
        df['modern_contraceptive'] = (df['v313'] == 3).astype(int)

        # Prepare predictors (adjust as needed based on available data)
        model_vars = ['v013', 'v106', 'v025', 'v190']
        available_vars = [var for var in model_vars if var in df.columns]

        if len(available_vars) >= 2:  # Need at least some predictors
            # Drop missing values
            model_df = df[['modern_contraceptive'] + available_vars].dropna()

            # Build formula
            formula = 'modern_contraceptive ~ ' + ' + '.join(available_vars)

            # Fit logistic regression
            try:
                model = smf.logit(formula=formula, data=model_df).fit(disp=0)
                print("\nLogistic Regression Results:")
                print(model.summary().tables[1])

                # Odds ratios
                print("\nOdds Ratios (effects on likelihood of using modern contraceptives):")
                odds_ratios = pd.DataFrame({
                    'Odds Ratio': np.exp(model.params),
                    '95% CI Lower': np.exp(model.conf_int()[0]),
                    '95% CI Upper': np.exp(model.conf_int()[1]),
                    'p-value': model.pvalues
                })
                print(odds_ratios)
            except:
                print("Could not fit logistic regression model with available variables")
        else:
            print("Not enough variables available for modeling")
    except:
        print("Could not create model for modern contraceptive use")


===== PREDICTIVE MODELING FOR SRHR OUTCOMES =====

MODEL 1: FACTORS AFFECTING MODERN CONTRACEPTIVE USE

Logistic Regression Results:
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.5483      0.050    -50.961      0.000      -2.646      -2.450
v013           0.2295      0.004     54.392      0.000       0.221       0.238
v106           0.3406      0.011     31.743      0.000       0.320       0.362
v025           0.1396      0.020      7.112      0.000       0.101       0.178
v190           0.1234      0.007     16.633      0.000       0.109       0.138

Odds Ratios (effects on likelihood of using modern contraceptives):
           Odds Ratio  95% CI Lower  95% CI Upper        p-value
Intercept    0.078215      0.070913      0.086269   0.000000e+00
v013         1.258029      1.247667      1.268478   0.000000e+00
v106         1.405780      1.376526      1.435656

 ## Predictive Modeling Results

The logistic regression model reveals:

### Age Group (v013):
 Each increase in age group is associated with 26% higher odds of using modern contraceptives (OR=1.26).

### Education (v106):
Each higher education level increases modern contraceptive use odds by 40.6% (OR=1.41).

### Urban Residence (v025):
 Urban women have 15% higher odds of using modern contraceptives than rural women (OR=1.15).
### Wealth Index (v190):
Each increase in wealth quintile is associated with 13.1% higher odds of modern contraceptive use (OR=1.13).

All predictors are highly statistically significant (p<0.001), with education having the strongest effect.